# Safety classifiers: work through F2, errors, and significant gaps

Start with the overview, then work through each question. All plots and tables use saved evaluations; **running this notebook makes no model calls**. Outputs are saved so you can read it before rerunning.

The main objects you can filter are `scores` (73 model/question/test configurations), `pairwise` (345 paired comparisons), `significant_gaps` (177 at adjusted p < .05), and `judgments` (individual decisions). Display formatting does not change their numeric values.

Use the repository's `.venv` Python kernel. Extra notebook dependencies: `uv pip install --python .venv/bin/python nbformat nbclient ipykernel jinja2`. Run all cells from top to bottom. This notebook finds the repository from either its root or the `analysis/` directory.

Every chart and table has a blank Markdown caption cell directly beneath it. Edit those cells to write your captions; rerunning code preserves them.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Image, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'analysis/plot_safety_f2.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the classifier_eval repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.source_report import NAMES, RUNS
from analysis.plot_safety_usecases import COHORTS, select
from analysis.braintrust_viz import use_braintrust_theme, clean_axes, rate_cmap, diverging_cmap
from analysis.f2_significance import estimate

%matplotlib inline
use_braintrust_theme()
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 110)
OUT = ROOT / 'analysis/output/safety-usecases-2026-10-14'
SOURCE = ROOT / 'analysis/output/source-2026-10-12'
ALPHA = 0.05  # Existing global Holm adjustment is preserved when filtering.
LABELS = {
    'aegis_single': 'Request alone · Aegis',
    'xstest_prompt': 'Request alone · XSTest',
    'wildguard_response': 'Assistant answer safety · WildGuardTest',
    'aegis_conversation': 'Whole exchange · Aegis',
    'wildguard_conversation': 'Either message unsafe · WildGuardTest',
    'wildguard_refusal': 'Refusal detection · WildGuardTest',
    'wildguard_prompt': 'Request with answer visible · WildGuardTest',
}
COHORT_ORDER = list(LABELS)
print(f'Repository: {ROOT}')


# Company identifies the model developer, not the gateway or hosting provider.
# Groups reflect the tested interface/model; they are not architecture claims.
BRANDING = {
    'clef_choice': ('Cloudflare', 'cloudflare', 'Decision'),
    'clef_flash_choice': ('Cloudflare', 'cloudflare', 'Decision'),
    'jev_choice': ('TypeSafe AI', 'typesafe', 'Decision'),
    'jev_noul': ('TypeSafe AI', 'typesafe', 'Decision'),
    'decisions_luna': ('OpenAI', 'openai', 'Decision'),
    'tev1_native': ('Together AI', 'together', 'Decision'),
    'claude_structured': ('Anthropic', 'anthropic', 'Text'),
    'luna_structured': ('OpenAI', 'openai', 'Text'),
    'qwen35_4b_letters': ('Alibaba / Qwen', 'qwen', 'Text'),
    'qwen35_4b_json': ('Alibaba / Qwen', 'qwen', 'Text'),
    'safeguard_120b': ('OpenAI', 'openai', 'Text'),
}
model_metadata = pd.DataFrame([
    {'arm': arm, 'model_name': NAMES[arm], 'company': company,
     'logo_key': logo, 'model_type': kind}
    for arm, (company, logo, kind) in BRANDING.items()
])
LOGO_DIR = ROOT / 'analysis/assets/company-logos'

from matplotlib.lines import Line2D
MODEL_STYLE = {
    'Decision': {'color': '#2C1FEB', 'hatch': None, 'edgecolor': 'white'},
    'Text': {'color': '#E6A4D3', 'hatch': '//', 'edgecolor': '#956BDD'},
}

## What F2 means here

$F_2 = 5TP / (5TP + 4FN + FP)$. Higher is better. The positive class is **unsafe** for safety questions and **refusal** for refusal detection. Missing decisions on positive examples count as FN. Missing decisions on negative examples do not count as FP; inspect failed decisions separately.

Compare models **within the same question and test**. Case mix and safety labels differ across tests. Aegis validation and test examples are pooled by example count; they are not assumed equivalent.

The F2 error bars are marginal 95% percentile bootstrap intervals (10,000 resamples). Significance uses paired delta-method tests on the same examples, followed by Holm correction across **all 345 comparisons**. Error-bar overlap is not the significance test. Tests assume independent examples and are conditional on these saved model runs; they do not include run-to-run variability. These are exploratory findings from already examined benchmarks, not confirmatory evidence on a fresh test set.

In [ ]:
f2_raw = pd.read_csv(OUT / 'f2_metrics.csv')
metrics = pd.read_csv(OUT / 'usecase_metrics.csv')
pairwise = pd.read_csv(OUT / 'f2_pairwise_significance.csv')
judgments = pd.read_csv(SOURCE / 'judgments.csv.gz', low_memory=False)

assert not f2_raw.duplicated(['arm', 'cohort']).any()
assert len(pairwise) == 345
assert judgments.job_id.is_unique
np.testing.assert_allclose(f2_raw.f2, 5*f2_raw.tp / (5*f2_raw.tp + 4*f2_raw.fn + f2_raw.fp))
assert pairwise.p_holm_all_345.between(0, 1).all()

# Join diagnostics by their exact configuration, without mixing policy modes or records.
identity = ['arm', 'cohort', 'model', 'provider', 'policy_mode', 'record_kind',
            'scheme', 'rubric', 'model_version', 'usecase']
rate_table = metrics.pivot(index=identity, columns='metric', values='rate').reset_index()
scores = f2_raw.merge(rate_table, on=identity, validate='one_to_one')
scores['model_name'] = scores.arm.map(NAMES)
scores['question'] = scores.cohort.map(LABELS)
scores['company'] = scores.arm.map(lambda arm: BRANDING[arm][0])
scores['model_type'] = scores.arm.map(lambda arm: BRANDING[arm][2])
scores['recall'] = scores.tp / (scores.tp + scores.fn)
scores['precision'] = scores.tp / (scores.tp + scores.fp).replace(0, np.nan)
scores['f1'] = 2*scores.tp / (2*scores.tp + scores.fn + scores.fp)
scores['no_decision_rate'] = scores.no_decision_n / scores.total_n
scores['f2_rank'] = scores.groupby('cohort').f2.rank(ascending=False, method='min')
scores['f1_rank'] = scores.groupby('cohort').f1.rank(ascending=False, method='min')
scores['rank_change_f1_to_f2'] = scores.f1_rank - scores.f2_rank
print(f'{len(scores)} configurations; {len(pairwise)} paired comparisons; '
      f'{pairwise.p_holm_all_345.lt(ALPHA).sum()} significant at adjusted p < {ALPHA}.')

## The overall data picture

The working claim is **“Clef is a strong broad safety classifier under F2, with the clearest advantage on whole-exchange safety.”** The evidence does not support “Clef wins every task” or “decision models always beat text models.” These comparisons mix repeated questions about the same examples; seven question/test combinations are not seven independent experiments.

The next plots separate three questions: how broad is the evidence for Clef, how convincing is its lead against the best alternative, and what quality do you get for the recorded usage cost?

In [ ]:
# Broad evidence for Clef, preserving the original all-345 Holm adjustment.
clef_story_records = []
for cohort in COHORT_ORDER:
    d = scores[scores.cohort.eq(cohort)].sort_values('f2', ascending=False)
    clef = d[d.arm.eq('clef_choice')].iloc[0]
    peer = d[d.arm.ne('clef_choice')].iloc[0]
    comparisons = pairwise[pairwise.cohort.eq(cohort) &
                          (pairwise.arm_a.eq('clef_choice') | pairwise.arm_b.eq('clef_choice'))].copy()
    differences = np.where(comparisons.arm_a.eq('clef_choice'), comparisons.difference, -comparisons.difference)
    significant = comparisons.p_holm_all_345.lt(ALPHA)
    top_pair = comparisons[comparisons.arm_a.eq(peer.arm) | comparisons.arm_b.eq(peer.arm)].iloc[0]
    sign = 1 if top_pair.arm_a == 'clef_choice' else -1
    ci = sorted([100*sign*top_pair.ci_low, 100*sign*top_pair.ci_high])
    clef_story_records.append({
        'cohort': cohort, 'question': LABELS[cohort], 'clef_f2': clef.f2,
        'clef_rank': int(d.arm.tolist().index('clef_choice')+1), 'models_tested': len(d),
        'best_alternative_arm': peer.arm, 'best_alternative': peer.model_name,
        'alternative_f2': peer.f2, 'gap_to_best_alternative_pp': 100*(clef.f2-peer.f2),
        'paired_ci_low_pp': ci[0], 'paired_ci_high_pp': ci[1],
        'p_adjusted_vs_best': top_pair.p_holm_all_345,
        'significant_wins': int((significant & (differences > 0)).sum()),
        'significant_losses': int((significant & (differences < 0)).sum()),
        'unresolved': int((~significant).sum()),
    })
clef_story = pd.DataFrame(clef_story_records)
assert (clef_story[['significant_wins','significant_losses','unresolved']].sum(axis=1)
        == clef_story.models_tested-1).all()
headline_counts = pd.Series({
    'Highest observed F2': int(clef_story.clef_rank.eq(1).sum()),
    'Question/test combinations': len(clef_story),
    'Significant wins': int(clef_story.significant_wins.sum()),
    'Significant losses': int(clef_story.significant_losses.sum()),
    'Unresolved comparisons': int(clef_story.unresolved.sum()),
})
display(headline_counts.to_frame('Count'))

**Table 1 caption:**



In [ ]:
display(clef_story[['question','clef_f2','clef_rank','models_tested','best_alternative',
                   'alternative_f2','gap_to_best_alternative_pp','p_adjusted_vs_best']]
        .style.format({'clef_f2':'{:.1%}','alternative_f2':'{:.1%}',
                       'gap_to_best_alternative_pp':'{:+.2f}','p_adjusted_vs_best':'{:.3g}'}))

**Table 2 caption:**



### 1. Clef's strength is broad, but it is not the leader on every question

Count each significant pair only within its own question/test combination. The outlined part is unresolved, not a tie. Ranks reflect observed F2 and need not be statistically distinguishable. Counts are descriptive summaries of the fixed comparison set, with no new significance tests on their totals.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.8))
y = np.arange(len(clef_story)); offset = np.zeros(len(clef_story))
for column, color, label in [('significant_wins','#2C1FEB','Clef significantly higher'),
                             ('unresolved','#F3F3F5','Unresolved'),
                             ('significant_losses','#A15B96','Clef significantly lower')]:
    values = clef_story[column].to_numpy()
    ax.barh(y, values, left=offset, height=.62, color=color,
            edgecolor='#B8B8C0' if column=='unresolved' else 'white', linewidth=.6, label=label)
    for pos,(width,start) in enumerate(zip(values,offset)):
        if width:
            ax.text(start+width/2,pos,str(width),va='center',ha='center',fontsize=12,
                    color='#18181B' if column=='unresolved' else 'white',fontweight='bold')
    offset += values
for pos,r in enumerate(clef_story.itertuples()):
    ax.text(offset[pos]+.25,pos,f'F2 rank {r.clef_rank}/{r.models_tested}',va='center',fontsize=11)
ax.set_yticks(y,clef_story.question,fontsize=11); ax.set_ylim(len(y)-.6,-.6)
ax.set_xlim(0,13);ax.set_xticks([0,2,4,6,8,10]);ax.set_xlabel('Number of other models in this question/test')
ax.set_title('Clef: broad strength across safety questions',loc='left',pad=54,fontsize=21,fontweight='bold')
ax.legend(loc='lower left',bbox_to_anchor=(0,1.015),ncol=3,frameon=False,fontsize=10)
clean_axes(ax,despine_left=True,grid_axis='x')
fig.tight_layout(pad=2.5)
fig.savefig(OUT/'09_clef_evidence_across_questions.png',dpi=200,bbox_inches='tight',pad_inches=.35)
display(fig);plt.close(fig)

**Figure 1 caption:**



### 2. Compare Clef with the strongest alternative, not just weaker models

Right of zero means higher observed F2 for Clef; left means higher F2 for the strongest non-Clef model in that test. Filled indigo marks a significant Clef advantage after global correction. Outlined bars are unresolved. Whiskers are **unadjusted paired 95% intervals**; filled/outlined status and p-values use the adjusted tests. A whisker can exclude zero while the globally adjusted result remains unresolved.

The alternative is selected from these observed data, so this is an exploratory comparison. There is no pooled F2 or average rank pretending to be a single evaluation outcome.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
y = np.arange(len(clef_story))
for pos,r in enumerate(clef_story.itertuples()):
    sig = r.p_adjusted_vs_best < ALPHA
    color = '#2C1FEB' if sig and r.gap_to_best_alternative_pp > 0 else '#A15B96' if sig else 'white'
    ax.barh(pos,r.gap_to_best_alternative_pp,height=.60,color=color,
            edgecolor=color if sig else '#9898A2',linewidth=1.2,zorder=2)
    ax.errorbar(r.gap_to_best_alternative_pp,pos,
                xerr=[[r.gap_to_best_alternative_pp-r.paired_ci_low_pp],
                      [r.paired_ci_high_pp-r.gap_to_best_alternative_pp]],
                fmt='none',ecolor='#18181B',capsize=4,elinewidth=1.2,zorder=3)
    p_label = '<.001' if r.p_adjusted_vs_best < .001 else f'{r.p_adjusted_vs_best:.3f}'
    ax.text(1.03,pos,f'{r.gap_to_best_alternative_pp:+.1f} pp vs {r.best_alternative}  |  p {p_label}',
            transform=ax.get_yaxis_transform(),va='center',fontsize=10)
ax.axvline(0,color='#18181B',linewidth=1)
ax.set_yticks(y,clef_story.question,fontsize=11);ax.set_ylim(len(y)-.6,-.6)
ax.set_xlabel('Clef minus strongest alternative (F2 percentage points)')
ax.set_title('One clear lead over the strongest alternative',loc='left',pad=20,fontsize=21,fontweight='bold')
clean_axes(ax,despine_left=True,grid_axis='x')
fig.tight_layout(pad=2.5)
fig.savefig(OUT/'10_clef_vs_best_alternative.png',dpi=200,bbox_inches='tight',pad_inches=.35)
display(fig);plt.close(fig)

**Figure 2 caption:**



### 3. Cost: use matched questions and recorded usage

These are **recorded token-usage estimates at the saved October 12, 2026 pricing snapshot**, not current quotes or reconciled invoices. They exclude unknown gateway fees, subscriptions, unrecorded retries, and infrastructure charges. Each mean includes successful and failed physical requests on the question's labeled examples. Models are matched on source dataset and example ID within each panel. Failures are not dropped to improve the apparent cost or F2.

Dedicated GPU models have no comparable token-billing estimate here. Their costs stay missing; they are not treated as free, and the runner-time allocation scenario is not mixed into the hosted API comparison. Do not compare total spend across unequal workloads.

In [ ]:
price_snapshot = json.loads((ROOT/'analysis/source-pricing-2026-10-12.json').read_text())
assert price_snapshot['checked_date'] == '2026-10-12'
cost_records = []
for cohort in COHORT_ORDER:
    g = select(judgments, COHORTS[cohort])
    g = g[g.policy_mode.eq('common_policy') & g.record_kind.eq('physical') & g.source_gold.notna()]
    ids_by_arm = {}
    for arm,d in g.groupby('arm'):
        if arm not in BRANDING:
            continue
        assert not d.duplicated(['source_dataset','item_id']).any()
        ids_by_arm[arm] = set(zip(d.source_dataset,d.item_id))
        n = len(d); priced_n = int(d.token_cost_usd.notna().sum())
        fully_priced = priced_n == n
        mean = float(d.token_cost_usd.mean())*1000 if fully_priced else np.nan
        se = float(d.token_cost_usd.std(ddof=1))/np.sqrt(n)*1000 if fully_priced else np.nan
        cost_records.append({'cohort':cohort,'arm':arm,'matched_n':n,'priced_n':priced_n,
            'price_coverage':priced_n/n,'estimated_usd_per_1k':mean,
            'cost_mean_ci_low':max(0,mean-1.96*se) if fully_priced else np.nan,
            'cost_mean_ci_high':mean+1.96*se if fully_priced else np.nan,
            'recorded_median_latency_s':d.latency_s.median(),
            'recorded_p95_latency_s':d.latency_s.quantile(.95),
            'cost_basis':'Recorded usage; saved 2026-10-12 token rates' if fully_priced else 'No comparable token-billing estimate'})
    reference_ids = next(iter(ids_by_arm.values()))
    assert all(ids == reference_ids for ids in ids_by_arm.values()), cohort
cost_quality = scores.merge(pd.DataFrame(cost_records),on=['cohort','arm'],validate='one_to_one')
assert cost_quality.total_n.eq(cost_quality.matched_n).all()
HOSTED_COST_COHORT = 'wildguard_response'
answer_costs = cost_quality[cost_quality.cohort.eq(HOSTED_COST_COHORT) & cost_quality.price_coverage.eq(1)].copy()
clef_cost = answer_costs.set_index('arm').loc['clef_choice','estimated_usd_per_1k']
answer_costs['cost_relative_to_clef'] = answer_costs.estimated_usd_per_1k/clef_cost
answer_costs['clef_cost_multiple'] = clef_cost/answer_costs.estimated_usd_per_1k
p_by_arm = {}
for arm in answer_costs.arm:
    if arm == 'clef_choice':
        p_by_arm[arm] = np.nan
    else:
        row = pairwise[pairwise.cohort.eq(HOSTED_COST_COHORT) &
                       ((pairwise.arm_a.eq(arm)&pairwise.arm_b.eq('clef_choice')) |
                        (pairwise.arm_b.eq(arm)&pairwise.arm_a.eq('clef_choice')))]
        assert len(row)==1
        p_by_arm[arm] = row.iloc[0].p_holm_all_345
answer_costs['p_adjusted_vs_clef'] = answer_costs.arm.map(p_by_arm)
COST_COLUMNS = ['model_name','f2','estimated_usd_per_1k','cost_relative_to_clef',
                'p_adjusted_vs_clef','recorded_median_latency_s','matched_n','price_coverage']
display(answer_costs[COST_COLUMNS].sort_values('estimated_usd_per_1k').style.format({
    'f2':'{:.1%}','estimated_usd_per_1k':'${:.3f}','cost_relative_to_clef':'{:.2f}×',
    'p_adjusted_vs_clef':'{:.3g}','recorded_median_latency_s':'{:.3f}', 'price_coverage':'{:.0%}'},na_rep='—'))

**Table 3 caption:**



In [ ]:
# Full cohort table keeps missing GPU cost values explicit.
cost_quality.to_csv(OUT/'cohort_cost_quality.csv',index=False)
answer_costs[COST_COLUMNS].to_csv(OUT/'answer_safety_cost_comparison.csv',index=False)
clef_story.to_csv(OUT/'clef_overall_evidence.csv',index=False)

### 4. The cost-quality frontier changes with the job

Each dot is a hosted API model on matched examples. The dashed line connects the **observed** cost/F2 frontier: a model is dominated if another has lower or equal estimated cost and higher or equal observed F2, with at least one strict improvement. This is descriptive, not a statistical test of dominance. Whiskers show marginal 95% F2 intervals and 95% normal-approximation intervals for mean usage cost over the sampled examples; the latter do not cover fees or pricing uncertainty. The logarithmic cost axis shows multiples rather than equal dollar increments.

For answer safety, Clef occupies the highest-quality end of the hosted frontier. Jev and Luna structured are cheaper candidates whose F2 gaps from Clef are unresolved after correction. On XSTest, Clef Flash is cheaper and has higher observed F2 than Clef, although that quality difference is unresolved.

In [ ]:
from matplotlib.ticker import FuncFormatter

fig, axes = plt.subplots(1,2,figsize=(17,7))
frontier_rows = []
for ax,cohort in zip(axes,['wildguard_response','xstest_prompt']):
    d = cost_quality[cost_quality.cohort.eq(cohort) & cost_quality.price_coverage.eq(1)].copy()
    d = d.sort_values('estimated_usd_per_1k')
    best = -np.inf; frontier = []
    for r in d.itertuples():
        if r.f2 > best:
            frontier.append(r);best=r.f2
    ax.plot([r.estimated_usd_per_1k for r in frontier],[r.f2 for r in frontier],
            color='#9898A2',linestyle='--',linewidth=1.2,zorder=1)
    frontier_arms={r.arm for r in frontier}
    for r in d.itertuples():
        kind=BRANDING[r.arm][2];color=MODEL_STYLE[kind]['color']
        ax.errorbar(r.estimated_usd_per_1k,r.f2,
                    xerr=[[r.estimated_usd_per_1k-r.cost_mean_ci_low],[r.cost_mean_ci_high-r.estimated_usd_per_1k]],
                    yerr=[[r.f2-r.ci_low],[r.ci_high-r.f2]],
                    fmt='D' if kind=='Decision' else 'o',color=color,ecolor='#956BDD',
                    markersize=8,capsize=3,elinewidth=1,zorder=3)
        offset = (-8,10) if r.arm=='claude_structured' else (8,8)
        alignment = 'right' if r.arm=='claude_structured' else 'left'
        if cohort=='wildguard_response' and r.arm=='decisions_luna':offset=(8,-18)
        if cohort=='wildguard_response' and r.arm=='luna_structured':offset=(-9,18);alignment='right'
        if cohort=='wildguard_response' and r.arm=='jev_choice':offset=(-9,10);alignment='right'
        if cohort=='xstest_prompt' and r.arm=='luna_structured':offset=(8,-23)
        if cohort=='xstest_prompt' and r.arm=='jev_choice':offset=(-9,-22);alignment='right'
        if cohort=='xstest_prompt' and r.arm=='clef_choice':offset=(8,-5)
        ax.annotate(r.model_name,(r.estimated_usd_per_1k,r.f2),xytext=offset,textcoords='offset points',
                    ha=alignment,fontsize=11,
                    bbox={'facecolor':'white','edgecolor':'none','pad':1})
        frontier_rows.append({'cohort':cohort,'model_name':r.model_name,'observed_frontier':r.arm in frontier_arms})
    ax.set_xscale('log')
    ax.set_xlim(d.estimated_usd_per_1k.min()*.60,d.estimated_usd_per_1k.max()*1.55)
    lower,upper=ax.get_xlim()
    ax.set_xticks([x for x in [.01,.03,.1,.3,1,3] if lower <= x <= upper])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f'${value:g}'))
    ax.set_ylim(max(0,d.ci_low.min()-.055),min(1,d.ci_high.max()+.06))
    ax.yaxis.set_major_formatter(PercentFormatter(1,decimals=0))
    ax.set_title(LABELS[cohort],loc='left',pad=18,fontsize=16)
    ax.set_xlabel('Estimated token usage cost (USD per 1,000 checks; log scale)')
    ax.set_ylabel('F2 score (higher is better)')
    ax.grid(True,axis='both',color='#E8E8E8')
    clean_axes(ax,grid_axis='both')
legend=[Line2D([],[],marker='D',linestyle='none',color='#2C1FEB',label='Decision model / interface'),
        Line2D([],[],marker='o',linestyle='none',color='#E6A4D3',label='Text model'),
        Line2D([],[],color='#9898A2',linestyle='--',label='Observed frontier')]
fig.legend(handles=legend,loc='upper center',ncol=3,frameon=False,fontsize=11)
fig.suptitle('Cost-quality tradeoffs depend on the task',x=.03,y=1.04,ha='left',fontsize=23,fontweight='bold')
fig.tight_layout(rect=(0,0,1,.94),pad=2.5)
fig.savefig(OUT/'11_hosted_cost_f2_frontiers.png',dpi=200,bbox_inches='tight',pad_inches=.35)
display(fig);plt.close(fig)

**Figure 3 caption:**



In [ ]:
frontier_table=pd.DataFrame(frontier_rows)
display(frontier_table)

**Table 4 caption:**



### 5. A missed unsafe case can be a wrong label or a failed decision

The stacked bars separate those mechanisms. All three categories count as misses in F2. Whiskers are 95% Wilson intervals for the **total** miss rate, not for each stacked segment. These plots describe errors; they do not establish why a model produced them.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(17,7))
main_hosted=['clef_choice','clef_flash_choice','jev_choice','decisions_luna','luna_structured','claude_structured']
for ax,cohort in zip(axes,['wildguard_response','wildguard_prompt']):
    d=metrics[metrics.cohort.eq(cohort) & metrics.arm.isin(main_hosted)]
    miss=d[d.metric.eq('positive_not_flagged')].set_index('arm').sort_values('rate')
    order=miss.index.tolist();y=np.arange(len(order));start=np.zeros(len(order))
    for metric,color,label in [('positive_wrong_negative','#956BDD','Called safe'),
                               ('positive_no_decision','#2C1FEB','Failed to give a decision'),
                               ('positive_other_valid','#E6A4D3','Other non-positive label')]:
        values=d[d.metric.eq(metric)].set_index('arm').loc[order].rate.to_numpy()
        ax.barh(y,values,left=start,height=.62,color=color,edgecolor='white',label=label)
        start+=values
    np.testing.assert_allclose(start,miss.rate)
    ax.errorbar(miss.rate,y,xerr=np.vstack([miss.rate-miss.ci_low,miss.ci_high-miss.rate]),
                fmt='none',ecolor='#18181B',capsize=4,zorder=4)
    ax.set_yticks(y,[NAMES[a] for a in order]);ax.invert_yaxis()
    ax.xaxis.set_major_formatter(PercentFormatter(1,decimals=0));ax.set_xlim(left=0)
    ax.set_xlabel('Share of unsafe examples missed (lower is better)')
    ax.set_title(LABELS[cohort],loc='left',pad=18,fontsize=15)
    clean_axes(ax,despine_left=True,grid_axis='x')
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',ncol=3,frameon=False,fontsize=11)
fig.suptitle('Separate wrong safety judgments from failed safety checks',x=.03,y=1.04,ha='left',fontsize=22,fontweight='bold')
fig.tight_layout(rect=(0,0,1,.94),pad=2.5)
fig.savefig(OUT/'12_miss_mechanisms.png',dpi=200,bbox_inches='tight',pad_inches=.35)
display(fig);plt.close(fig)

**Figure 4 caption:**



### Suggested takeaways for the report

The following summary is generated from the saved tables so the numerical claims stay tied to the data. Cost ratios describe usage estimates at the saved rates, not current billing or expected production spend. Cheaper candidates with unresolved F2 gaps need fresh representative tests; unresolved does not demonstrate equivalent quality.

In [ ]:
answer_lookup=answer_costs.set_index('arm')
clef_answer=answer_lookup.loc['clef_choice']; claude_answer=answer_lookup.loc['claude_structured']
jev_answer=answer_lookup.loc['jev_choice']; luna_answer=answer_lookup.loc['luna_structured']
wg_story=clef_story.set_index('cohort').loc['wildguard_conversation']
text = f"""
**Broad quality:** Clef has the highest observed F2 on {headline_counts['Highest observed F2']} of {headline_counts['Question/test combinations']} question/test combinations. Across its {headline_counts['Significant wins']+headline_counts['Significant losses']+headline_counts['Unresolved comparisons']} within-question comparisons, it has {headline_counts['Significant wins']} significant wins, {headline_counts['Significant losses']} significant losses, and {headline_counts['Unresolved comparisons']} unresolved gaps under the global Holm correction. These are correlated comparisons, not independent replications.

**Clearest win:** On WildGuardTest whole exchanges, Clef scores {wg_story.clef_f2:.1%} F2 and beats every tested competitor. Its lead over the strongest alternative, {wg_story.best_alternative}, is {wg_story.gap_to_best_alternative_pp:.1f} points (adjusted p = {wg_story.p_adjusted_vs_best:.3f}).

**Task-specific exceptions:** Claude has the highest observed F2 on XSTest request screening; Safeguard leads refusal detection; Qwen JSON narrowly leads Aegis single-message safety. None significantly beats Clef after the global correction. F2's emphasis on positive-case recall is part of this conclusion; false alarms and failed decisions still matter.

**Cost versus Claude:** On the same 1,681 answer-safety examples, Clef's estimated token usage cost is ${clef_answer.estimated_usd_per_1k:.3f} per 1,000 checks versus ${claude_answer.estimated_usd_per_1k:.3f} for Claude: {claude_answer.estimated_usd_per_1k/clef_answer.estimated_usd_per_1k:.1f}× lower estimated cost. Clef also has {100*(clef_answer.f2-claude_answer.f2):.1f} points higher F2; that quality gap is significant after correction.

**Cost versus cheaper alternatives:** Jev is {clef_answer.estimated_usd_per_1k/jev_answer.estimated_usd_per_1k:.1f}× cheaper than Clef on answer safety (${jev_answer.estimated_usd_per_1k:.3f}/1,000 checks) at {jev_answer.f2:.1%} observed F2; Luna structured is {clef_answer.estimated_usd_per_1k/luna_answer.estimated_usd_per_1k:.1f}× cheaper (${luna_answer.estimated_usd_per_1k:.3f}/1,000) at {luna_answer.f2:.1%}. Clef's observed F2 is {clef_answer.f2:.1%}; its gaps from both are unresolved after correction. This supports testing cheaper options, not claiming they are equivalent.

**Practical conclusion:** Treat Clef as a strong safety-quality candidate, especially for whole exchanges. Treat cost as a task-specific tradeoff: Clef is much cheaper than Claude here, but it is not the cheapest capable option. Use a fresh workload with your policy and operating costs to select a deployment.
"""
display(Markdown(text))

In [ ]:
key_takeaways=pd.DataFrame([
    {'theme':'Broad quality','finding':f"Highest observed F2 on {headline_counts['Highest observed F2']}/7 combinations; {headline_counts['Significant wins']} significant wins, {headline_counts['Significant losses']} significant losses",'qualification':'Exploratory; correlated questions and examples; F2 objective'},
    {'theme':'Clearest advantage','finding':f'Whole-exchange WildGuardTest: +{wg_story.gap_to_best_alternative_pp:.1f} pp versus strongest alternative','qualification':f'Global Holm-adjusted p = {wg_story.p_adjusted_vs_best:.3f}'},
    {'theme':'Cost versus Claude','finding':f'{claude_answer.estimated_usd_per_1k/clef_answer.estimated_usd_per_1k:.1f}× lower estimated token cost on answer safety','qualification':'Recorded usage at 2026-10-12 saved rates; not invoices'},
    {'theme':'Cheaper candidates','finding':f'Jev {clef_answer.estimated_usd_per_1k/jev_answer.estimated_usd_per_1k:.1f}× cheaper; Luna structured {clef_answer.estimated_usd_per_1k/luna_answer.estimated_usd_per_1k:.1f}× cheaper','qualification':'Quality gaps from Clef unresolved, not equivalent'},
])
display(key_takeaways)

**Table 5 caption:**



In [ ]:
key_takeaways.to_csv(OUT/'overall_key_takeaways.csv',index=False)

## 1. Overview: who does well at which question?

This overview is a saved figure; the per-question charts below are generated from the editable DataFrames. Blank cells mean not tested, not zero performance.

In [ ]:
display(Image(filename=str(OUT / '06_f2_overview.png'), width=1200))

**Figure 5 caption:**



In [ ]:
f2_table = scores.pivot(index='model_name', columns='question', values='f2').reindex(columns=list(LABELS.values()))
display(f2_table.style.format('{:.1%}', na_rep='Not tested'))

**Table 6 caption:**



### All model scores as a pandas table

`precision`, `recall`, and F2 use the positive-flag convention described above. `accuracy` also penalizes failed decisions on negative cases, so it can tell a different story from F2.

In [ ]:
SCORE_COLUMNS = ['question', 'model_name', 'company', 'model_type', 'total_n', 'positive_n', 'tp', 'fn', 'fp',
                 'precision', 'recall', 'f1', 'f2', 'ci_low', 'ci_high',
                 'accuracy', 'negative_false_alarm', 'no_decision_n']
score_table = scores[SCORE_COLUMNS].sort_values(['question', 'f2'], ascending=[True, False]).reset_index(drop=True)
with pd.option_context('display.max_rows', None):
    display(score_table.style.format({c: '{:.1%}' for c in
        ['precision', 'recall', 'f1', 'f2', 'ci_low', 'ci_high', 'accuracy', 'negative_false_alarm']}))

**Table 7 caption:**



## 2. F2 by question

Bar length shows F2; whiskers show the marginal 95% bootstrap intervals. **Indigo / solid = decision model or interface; pink / hatched = text model.** Logos identify the model developer rather than its inference host. The metadata table below makes every assignment editable.

Luna structured and Decisions Luna share the same base model and use different APIs. Safeguard is a safety-tuned text generator, so it is included in the text group. Tev1 is a decision-tuned model despite emitting a letter. These groups describe the evaluated arms, not a controlled causal comparison of architectures.

Logos are optional local assets and are not distributed with this repository. Charts use model names when no local logo is available. Public availability of a logo does not establish redistribution rights. Qwen uses its own brand mark under Alibaba / Qwen. Sources for the decision-model grouping: [Cloudflare](https://blog.cloudflare.com/clef-decision-models/), [TypeSafe](https://typesafe.ai/), [Together AI](https://www.together.ai/models/tev1-4b-experimental); the OpenAI interface distinction is recorded in `adapters/README.md`.

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.patches import Patch
import matplotlib.image as mpimg

def load_logo(key):
    path = LOGO_DIR / f'{key}.png'
    if not path.exists():
        return None  # Optional local artwork; model names remain visible.
    image = mpimg.imread(path)
    # Ignore empty white/transparent padding so each brand mark has the same visible size.
    foreground = np.any(image[..., :3] < .96, axis=2)
    if image.shape[2] == 4:
        foreground &= image[..., 3] > .05
    rows, cols = np.where(foreground)
    if rows.size:
        image = image[max(0, rows.min()-2):rows.max()+3, max(0, cols.min()-2):cols.max()+3]
    return image

logos = {key: load_logo(key) for key in {logo for _, logo, _ in BRANDING.values()}}

def plot_f2(cohorts, models=None):
    fig, axes = plt.subplots(1, len(cohorts), figsize=(9.5*len(cohorts), 6.5), squeeze=False)
    for ax, cohort in zip(axes[0], cohorts):
        data = scores[scores.cohort.eq(cohort)].copy()
        if models is not None:
            data = data[data.arm.isin(models)]
        data = data.sort_values('f2')
        if data.empty:
            raise ValueError(f'No selected models in {cohort}')
        y = np.arange(len(data))
        for pos, row in zip(y, data.itertuples()):
            company, logo_key, kind = BRANDING[row.arm]
            style = MODEL_STYLE[kind]
            ax.barh(pos, row.f2, height=.65, color=style['color'],
                    hatch=style['hatch'], edgecolor=style['edgecolor'], linewidth=.5, zorder=2)
            # Equal physical logo height and a fixed label column keep rows aligned.
            image = logos[logo_key]
            if image is not None:
                logo = OffsetImage(image, zoom=18/image.shape[0])
                ax.add_artist(AnnotationBbox(logo, (0, pos), xycoords=('axes fraction', 'data'),
                              xybox=(-157, 0), boxcoords='offset points',
                              frameon=False, annotation_clip=False))
            ax.annotate(row.model_name, xy=(0, pos), xycoords=('axes fraction', 'data'),
                        xytext=(-138, 0), textcoords='offset points',
                        ha='left', va='center', fontsize=11, annotation_clip=False)
            ax.text(1.025, pos, f'{row.f2:.1%}', va='center', fontsize=11, fontweight='bold')
        ax.errorbar(data.f2, y, xerr=np.vstack([data.f2-data.ci_low, data.ci_high-data.f2]),
                    fmt='none', ecolor='#18181B', capsize=4, elinewidth=1.5, zorder=3)
        ax.set_yticks(y, ['']*len(data))
        ax.set_title(f'{LABELS[cohort]}\nn = {int(data.total_n.iloc[0]):,}', loc='left', pad=18)
        ax.set_xlim(0, 1.01)
        ax.set_xticks([0, .25, .5, .75, 1])
        ax.xaxis.set_major_formatter(PercentFormatter(1))
        ax.set_xlabel('F2 score (higher is better)')
        ax.tick_params(axis='y', length=0)
        clean_axes(ax, despine_left=True, grid_axis='x')
    legend = [Patch(facecolor=style['color'], edgecolor=style['edgecolor'],
                    hatch=style['hatch'], label=label)
              for kind, label in [('Decision','Decision model / interface'), ('Text','Text model')]
              for style in [MODEL_STYLE[kind]]]
    fig.legend(handles=legend, loc='upper center', bbox_to_anchor=(.56, 1), ncol=2,
               frameon=False, fontsize=11)
    fig.tight_layout(pad=2.5, rect=(0,0,1,.91))
    display(fig)
    plt.close(fig)

display(model_metadata[['model_name', 'company', 'model_type']])

**Table 8 caption:**



### Request screening

Aegis and XSTest can rank the same models differently. On XSTest, check how false alarms affect a model that catches most unsafe requests.

In [ ]:
plot_f2(['aegis_single', 'xstest_prompt'])

**Figure 6 caption:**



### Assistant answer safety

This is the clearest example of accuracy hiding unsafe-answer misses. Clef has the highest observed F2, but its smaller gaps against several competitors do not survive the global correction.

In [ ]:
plot_f2(['wildguard_response'])

**Figure 7 caption:**



### Whole exchanges

WildGuardTest marks the exchange unsafe if either message is unsafe. That is a different target from answer safety. Clef’s advantage over Qwen letters survives correction here.

In [ ]:
plot_f2(['aegis_conversation', 'wildguard_conversation'])

**Figure 8 caption:**



### Refusal detection

Detecting a refusal is not the same as detecting a safe answer. The positive label in this panel is refusal.

In [ ]:
plot_f2(['wildguard_refusal'])

**Figure 9 caption:**



### Request safety with the answer visible

These models can see both messages, but are scored on the request. These results do not establish how well the model screens requests before an answer exists.

In [ ]:
plot_f2(['wildguard_prompt'])

**Figure 10 caption:**



## 3. What is driving the scores?

The most useful diagnostics distinguish misses, false alarms, and failures to produce a decision. The original charts retain separate rate intervals and denominators.

In [ ]:
# Change the filename to another original question plot as needed.
display(Image(filename=str(OUT / '02_response_safety.png'), width=1300))

**Figure 11 caption:**



In [ ]:
display(Image(filename=str(OUT / '04_refusal_detection.png'), width=1300))

**Figure 12 caption:**



### Does emphasizing recall change the ranking?

F1 and F2 are computed from exactly the same saved decisions. A positive `rank_change_f1_to_f2` means the model moves up when recall receives more emphasis. No cross-dataset average is used.

In [ ]:
RANK_COHORT = 'wildguard_response'  # Edit this.
rank_table = scores[scores.cohort.eq(RANK_COHORT)][
    ['model_name', 'precision', 'recall', 'f1', 'f2', 'f1_rank', 'f2_rank', 'rank_change_f1_to_f2']
].sort_values('f2_rank')
display(rank_table.style.format({c: '{:.1%}' for c in ['precision', 'recall', 'f1', 'f2']}))

**Table 9 caption:**



### Failed decisions: which misses are actually failures to judge?

A failed safety check needs operational handling. F2 penalizes failures on positive examples but does not penalize negative abstentions as false alarms. This chart counts **all** failed decisions and shows their Wilson intervals.

In [ ]:
FAILURE_COHORT = 'wildguard_response'
failed = metrics[metrics.cohort.eq(FAILURE_COHORT) & metrics.metric.eq('no_decision')].copy()
failed['model_name'] = failed.arm.map(NAMES)
failed = failed.sort_values('rate')
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(failed))
ax.errorbar(failed.rate, y, xerr=np.vstack([failed.rate-failed.ci_low, failed.ci_high-failed.rate]),
            fmt='o', color='#2C1FEB', ecolor='#956BDD', capsize=4)
ax.set_yticks(y, failed.model_name)
ax.set_title('Failed decisions: ' + LABELS[FAILURE_COHORT], loc='left', pad=18)
ax.set_xlabel('Share of all examples (lower is better)')
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=1))
clean_axes(ax, despine_left=True, grid_axis='x')
fig.tight_layout(); display(fig); plt.close(fig)

**Figure 13 caption:**



In [ ]:
display(failed[['model_name', 'k', 'n', 'rate', 'classifier_refusal_n', 'truncated_n', 'invalid_output_n']]
        .style.format({'rate': '{:.1%}'}))

**Table 10 caption:**



## 4. Every significant gap, as a pandas table

Higher/lower models are oriented consistently. `gap_pp` is the F2 difference in **percentage points**, not a relative percentage improvement. `p_adjusted` is corrected across all 345 comparisons, including ones not shown here. Filtering this table never recomputes a smaller correction family.

The `significant_gaps` DataFrame contains every significant pair. The next cell displays all rows, without pandas truncation.

In [ ]:
pairwise['question'] = pairwise.cohort.map(LABELS)
pairwise['a_higher'] = pairwise.difference.ge(0)
gaps = pd.DataFrame({
    'cohort': pairwise.cohort,
    'question': pairwise.question,
    'higher_arm': np.where(pairwise.a_higher, pairwise.arm_a, pairwise.arm_b),
    'lower_arm': np.where(pairwise.a_higher, pairwise.arm_b, pairwise.arm_a),
    'higher_f2': np.maximum(pairwise.f2_a, pairwise.f2_b),
    'lower_f2': np.minimum(pairwise.f2_a, pairwise.f2_b),
    'gap_pp': pairwise.difference.abs()*100,
    'p_adjusted': pairwise.p_holm_all_345,
    'n': pairwise.n,
})
gaps['higher_model'] = gaps.higher_arm.map(NAMES)
gaps['lower_model'] = gaps.lower_arm.map(NAMES)
gaps['significant'] = gaps.p_adjusted.lt(ALPHA)
gaps = gaps.sort_values(['question', 'gap_pp'], ascending=[True, False]).reset_index(drop=True)
significant_gaps = gaps[gaps.significant].copy().reset_index(drop=True)
GAP_COLUMNS = ['question', 'higher_model', 'lower_model', 'higher_f2', 'lower_f2', 'gap_pp', 'p_adjusted', 'n']

def show_gaps(frame):
    with pd.option_context('display.max_rows', None):
        display(frame[GAP_COLUMNS].style.format({'higher_f2':'{:.1%}', 'lower_f2':'{:.1%}',
                                                 'gap_pp':'{:.2f}', 'p_adjusted':'{:.3g}'}))

show_gaps(significant_gaps)

**Table 11 caption:**



### Filter a question, model, or minimum gap

Edit these values and rerun this cell. `FOCUS_MODEL` accepts an exact model name from `NAMES.values()`. Set filters to `None` to keep everything.

In [ ]:
FOCUS_COHORT = 'wildguard_response'  # Set to None for all questions.
FOCUS_MODEL = 'Clef'             # Set to None for all models.
MIN_GAP_PP = 0.0
filtered_gaps = significant_gaps.copy()
if FOCUS_COHORT is not None:
    filtered_gaps = filtered_gaps[filtered_gaps.cohort.eq(FOCUS_COHORT)]
if FOCUS_MODEL is not None:
    filtered_gaps = filtered_gaps[filtered_gaps.higher_model.eq(FOCUS_MODEL) | filtered_gaps.lower_model.eq(FOCUS_MODEL)]
filtered_gaps = filtered_gaps[filtered_gaps.gap_pp.ge(MIN_GAP_PP)]
print(f'{len(filtered_gaps)} / {len(significant_gaps)} significant gaps shown')
show_gaps(filtered_gaps)

**Table 12 caption:**



### Is the observed leader distinguishable from the runner-up?

The top two are chosen by observed F2 for each test. A non-significant result is not evidence that the models are equivalent.

In [ ]:
leader_rows = []
for cohort in COHORT_ORDER:
    best = scores[scores.cohort.eq(cohort)].nlargest(2, 'f2')
    a, b = best.arm.iloc[0], best.arm.iloc[1]
    match = gaps[gaps.cohort.eq(cohort) & gaps.higher_arm.eq(a) & gaps.lower_arm.eq(b)]
    assert len(match) == 1
    leader_rows.append(match.iloc[0])
leader_comparisons = pd.DataFrame(leader_rows).reset_index(drop=True)
display(leader_comparisons[GAP_COLUMNS + ['significant']].style.format({
    'higher_f2':'{:.1%}', 'lower_f2':'{:.1%}', 'gap_pp':'{:.2f}', 'p_adjusted':'{:.3g}'}))

**Table 13 caption:**



### Read the ranking, then compare against one model

This replaces the pairwise matrix with two aligned views. The left panel shows observed F2, company logos, and model type. The right panel answers **how much better or worse is each model than the reference?** Solid bars pass the existing global Holm correction; outlined bars do not. Each row includes the adjusted p-value. The reference is highlighted and has no test against itself.

Change `COMPARE_COHORT` and `REFERENCE_MODEL` below. Start with Qwen letters: on WildGuardTest whole exchanges, Clef is significantly higher, while Tev1 and Luna structured have lower observed F2 without a significant difference from Qwen letters after correction.

F2 whiskers on the left are marginal 95% bootstrap intervals. The right shows point differences and the **paired, globally adjusted** test results; no interval-overlap rule is used. “Unresolved” means not significant, not equal. The two panels use different color keys, labeled above each panel.

In [ ]:
from matplotlib.lines import Line2D

def comparison_view(cohort, reference):
    ranking = scores[scores.cohort.eq(cohort)].sort_values('f2', ascending=False).copy()
    if reference not in set(ranking.arm):
        raise ValueError(f'{reference} was not tested in {cohort}')
    baseline = ranking.set_index('arm').loc[reference]
    records = []
    for r in ranking.itertuples():
        if r.arm == reference:
            records.append({'arm': r.arm, 'model_name': r.model_name, 'f2': r.f2,
                            'gap_pp': 0., 'p_adjusted': np.nan, 'result': 'Reference'})
            continue
        pair = pairwise[pairwise.cohort.eq(cohort) &
                        ((pairwise.arm_a.eq(r.arm) & pairwise.arm_b.eq(reference)) |
                         (pairwise.arm_b.eq(r.arm) & pairwise.arm_a.eq(reference)))]
        assert len(pair) == 1
        pvalue = pair.iloc[0].p_holm_all_345
        delta = 100*(r.f2-baseline.f2)
        result = ('Higher' if delta > 0 else 'Lower') if pvalue < ALPHA else 'Unresolved'
        records.append({'arm': r.arm, 'model_name': r.model_name, 'f2': r.f2,
                        'gap_pp': delta, 'p_adjusted': pvalue, 'result': result})
    comparisons = pd.DataFrame(records)
    fig, (left, right) = plt.subplots(1, 2, figsize=(16, 8), sharey=True,
                                     gridspec_kw={'width_ratios': [1, 1.15]})
    fig.subplots_adjust(left=.19, right=.81, top=.70, bottom=.19, wspace=.30)
    y = np.arange(len(ranking))
    for pos, r in zip(y, ranking.itertuples()):
        kind = BRANDING[r.arm][2]
        style = MODEL_STYLE[kind]
        if r.arm == reference:
            for ax in (left, right):
                ax.axhspan(pos-.45, pos+.45, color='#F1EEFA', zorder=0)
        left.barh(pos, r.f2, height=.62, color=style['color'], hatch=style['hatch'],
                  edgecolor=style['edgecolor'], linewidth=.5, zorder=2)
        image = logos[BRANDING[r.arm][1]]
        left.add_artist(AnnotationBbox(OffsetImage(image, zoom=18/image.shape[0]),
                        (0,pos), xycoords=('axes fraction','data'), xybox=(-157,0),
                        boxcoords='offset points', frameon=False, annotation_clip=False))
        left.annotate(r.model_name, (0,pos), xycoords=('axes fraction','data'),
                      xytext=(-138,0), textcoords='offset points', ha='left', va='center',
                      fontsize=11, fontweight='bold' if r.arm == reference else 'normal', annotation_clip=False)
        left.text(r.f2+.018, pos, f'{r.f2:.1%}', va='center', fontsize=10,
                  fontweight='bold', bbox={'facecolor':'white','edgecolor':'none','pad':1})
    left.errorbar(ranking.f2, y, xerr=np.vstack([ranking.f2-ranking.ci_low, ranking.ci_high-ranking.f2]),
                  fmt='none', ecolor='#18181B', capsize=3, zorder=3)
    # Put score labels beyond the upper whisker rather than over it.
    for text, high in zip(left.texts[1::2], ranking.ci_high):
        text.set_x(high+.018)
    colors = {'Higher':'#2C1FEB', 'Lower':'#A15B96', 'Unresolved':'#FFFFFF'}
    for pos, row in zip(y, comparisons.itertuples()):
        if row.result == 'Reference':
            right.plot(0,pos,marker='D',color='#18181B',markersize=7,zorder=4)
            result_label = 'Reference'
        else:
            significant = row.result != 'Unresolved'
            right.barh(pos, row.gap_pp, height=.62, color=colors[row.result],
                       edgecolor=colors[row.result] if significant else '#9898A2',
                       linewidth=1.1, zorder=2)
            offset = .45 if row.gap_pp >= 0 else -.45
            right.text(row.gap_pp+offset,pos,f'{row.gap_pp:+.1f}',
                       ha='left' if row.gap_pp >= 0 else 'right',va='center',fontsize=11)
            p_label = '<.001' if row.p_adjusted < .001 else f'{row.p_adjusted:.3f}'
            result_label = f'{row.result}  ·  p {p_label}'
        right.text(1.03,pos,result_label,transform=right.get_yaxis_transform(),
                   va='center',fontsize=10,color='#18181B')
    for ax in (left,right):
        ax.set_yticks(y,['']*len(y)); ax.set_ylim(len(y)-.6,-.6)
        ax.tick_params(axis='y',length=0)
        clean_axes(ax,despine_left=True,grid_axis='x')
    left.set_xlim(0,1.13); left.set_xticks([0,.25,.5,.75,1])
    left.xaxis.set_major_formatter(PercentFormatter(1,decimals=0))
    left.set_xlabel('Observed F2 score')
    left.set_title('1. Overall F2 ranking',loc='left',pad=58,fontsize=15,fontweight='bold')
    left.legend(handles=[Patch(facecolor=MODEL_STYLE[k]['color'],hatch=MODEL_STYLE[k]['hatch'],
                               edgecolor=MODEL_STYLE[k]['edgecolor'],label=label)
                         for k,label in [('Decision','Decision'),('Text','Text')]],
                loc='lower left',bbox_to_anchor=(0,1.015),ncol=2,frameon=False,fontsize=10)
    lo,hi=comparisons.gap_pp.min(),comparisons.gap_pp.max()
    padding=max((hi-lo)*.17,2)
    right.set_xlim(min(0,lo)-padding,max(0,hi)+padding)
    right.axvline(0,color='#18181B',linewidth=1,zorder=1)
    right.set_xlabel(f'F2 gap vs {NAMES[reference]} (percentage points)')
    right.set_title('2. Difference from the reference',loc='left',pad=58,fontsize=15,fontweight='bold')
    right.legend(handles=[Patch(facecolor='#2C1FEB',label='Higher'),Patch(facecolor='#A15B96',label='Lower'),
                          Patch(facecolor='white',edgecolor='#9898A2',label='Unresolved')],
                 loc='lower left',bbox_to_anchor=(0,1.015),ncol=3,frameon=False,fontsize=10,
                 handlelength=1.3,columnspacing=1)
    counts=comparisons.result.value_counts()
    fig.text(.035,.95,LABELS[cohort],fontsize=23,fontweight='bold')
    fig.text(.035,.90,f'Reference: {NAMES[reference]} ({baseline.f2:.1%} F2)  |  '
                      f'{counts.get("Higher",0)} significantly higher · '
                      f'{counts.get("Lower",0)} significantly lower · '
                      f'{counts.get("Unresolved",0)} unresolved',fontsize=13,color='#6B6B76')
    fig.text(.035,.055,'Significance: paired tests, Holm-adjusted across all 345 comparisons, p < '
                       f'{ALPHA}. Unresolved does not mean equivalent.',fontsize=11,color='#6B6B76')
    display(fig)
    plt.close(fig)
    return comparisons, fig

COMPARE_COHORT = 'wildguard_conversation'
REFERENCE_MODEL = 'qwen35_4b_letters'  # Try 'clef_choice', 'luna_structured', or another arm ID.
reference_comparisons, reference_figure = comparison_view(COMPARE_COHORT, REFERENCE_MODEL)

**Figure 14 caption:**



In [ ]:
display(reference_comparisons[['model_name','f2','gap_pp','p_adjusted','result']].style.format(
    {'f2':'{:.1%}','gap_pp':'{:+.2f}','p_adjusted':'{:.3g}'},na_rep='—'))

**Table 14 caption:**



### All comparisons: who clearly beats whom?

This summarizes every adjusted test for the selected question. Bars to the right count models significantly below the row model; bars to the left count models significantly above it. The last column counts unresolved comparisons. Rows stay in F2 order.

These counts summarize evidence across this model set, not independent trials or a new quality metric. Pairwise significance is not transitive and does not partition models into clean tiers. Use the reference comparison above or `significant_gaps` to identify the individual pairs.

In [ ]:
# Follows COMPARE_COHORT from the previous cell.
model_order = scores[scores.cohort.eq(COMPARE_COHORT)].sort_values('f2', ascending=False).arm.tolist()
evidence = pd.DataFrame(0, index=pd.Index(model_order, name='arm'), columns=['higher_than', 'lower_than', 'unresolved'])
for row in pairwise[pairwise.cohort.eq(COMPARE_COHORT)].itertuples():
    if row.p_holm_all_345 < ALPHA:
        higher, lower = (row.arm_a, row.arm_b) if row.difference > 0 else (row.arm_b, row.arm_a)
        evidence.loc[higher, 'higher_than'] += 1
        evidence.loc[lower, 'lower_than'] += 1
    else:
        evidence.loc[[row.arm_a,row.arm_b], 'unresolved'] += 1
assert evidence.sum(axis=1).eq(len(model_order)-1).all()
assert evidence.higher_than.sum() == evidence.lower_than.sum()
evidence['model_name'] = evidence.index.map(NAMES)
fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(evidence))
ax.barh(y, evidence.higher_than, height=.65, color='#2C1FEB', label='Significantly higher than')
ax.barh(y, -evidence.lower_than, height=.65, color='#A15B96', label='Significantly lower than')
for pos, r in zip(y, evidence.itertuples()):
    if r.higher_than:
        ax.text(r.higher_than+.15,pos,str(r.higher_than),va='center',ha='left',fontsize=11)
    if r.lower_than:
        ax.text(-r.lower_than-.15,pos,str(r.lower_than),va='center',ha='right',fontsize=11)
    ax.text(1.04,pos,str(r.unresolved),transform=ax.get_yaxis_transform(),va='center',ha='center',fontsize=11)
ax.text(1.04,1.04,'Unresolved',transform=ax.transAxes,ha='center',fontsize=11)
ax.set_yticks(y,evidence.model_name)
ax.set_ylim(len(evidence)-.6,-.6)
maximum=len(model_order)-1
ax.set_xlim(-maximum-1,maximum+1)
ax.set_xticks(np.arange(-maximum,maximum+1,2))
ax.set_xticklabels([str(abs(x)) for x in ax.get_xticks()])
ax.axvline(0,color='#18181B',linewidth=1)
ax.set_xlabel('Number of other models in the selected test')
ax.set_title('Significant wins and losses: ' + LABELS[COMPARE_COHORT],loc='left',pad=48,fontsize=17)
ax.legend(loc='lower left',bbox_to_anchor=(0,1.01),ncol=2,frameon=False,fontsize=10)
clean_axes(ax,despine_left=True,grid_axis='x')
fig.tight_layout(pad=2.5)
display(fig); plt.close(fig)

**Figure 15 caption:**



In [ ]:
display(evidence[['model_name','higher_than','lower_than','unresolved']].reset_index(drop=True))

**Table 15 caption:**



## 5. Choose a pair and examine the actual decisions

Change the cohort and two model IDs below. This returns the existing globally adjusted test and exact paired outcomes, preserving the example join. The default is the significant Clef versus Qwen letters gap on WildGuardTest whole exchanges.

`a_only_correct` / `b_only_correct` describe exact source-label correctness; they are not a replacement for the paired F2 test. Two models can both be wrong for different reasons, so `decision_differs` is retained too.

In [ ]:
PAIR_COHORT = 'wildguard_conversation'
MODEL_A = 'clef_choice'
MODEL_B = 'qwen35_4b_letters'

cohort_rows = select(judgments, COHORTS[PAIR_COHORT])
cohort_rows = cohort_rows[cohort_rows.policy_mode.eq('common_policy') &
                          cohort_rows.record_kind.eq('physical') & cohort_rows.source_gold.notna()]
keys = ['source_dataset', 'item_id']
keep = keys + ['source_gold', 'label', 'status']
a = cohort_rows[cohort_rows.arm.eq(MODEL_A)][keep]
b = cohort_rows[cohort_rows.arm.eq(MODEL_B)][keep]
assert len(a) and len(b), 'Choose two models tested on this cohort.'
paired_examples = a.merge(b, on=keys, suffixes=('_a','_b'), validate='one_to_one', how='outer', indicator=True)
assert paired_examples._merge.eq('both').all(), 'Models must have matched examples.'
assert paired_examples.source_gold_a.eq(paired_examples.source_gold_b).all()
paired_examples['gold'] = paired_examples.source_gold_a
paired_examples['a_correct'] = paired_examples.status_a.eq('ok') & paired_examples.label_a.eq(paired_examples.gold)
paired_examples['b_correct'] = paired_examples.status_b.eq('ok') & paired_examples.label_b.eq(paired_examples.gold)
paired_examples['decision_differs'] = (paired_examples.label_a.fillna('<missing>').ne(paired_examples.label_b.fillna('<missing>')) |
                                        paired_examples.status_a.ne(paired_examples.status_b))
paired_examples['paired_outcome'] = np.select([
    paired_examples.a_correct & ~paired_examples.b_correct,
    ~paired_examples.a_correct & paired_examples.b_correct,
    paired_examples.a_correct & paired_examples.b_correct],
    ['a_only_correct','b_only_correct','both_correct'], default='both_wrong')
comparison = gaps[gaps.cohort.eq(PAIR_COHORT) &
                  ((gaps.higher_arm.eq(MODEL_A) & gaps.lower_arm.eq(MODEL_B)) |
                   (gaps.higher_arm.eq(MODEL_B) & gaps.lower_arm.eq(MODEL_A)))]
show_gaps(comparison)

**Table 16 caption:**



In [ ]:
display(pd.crosstab(paired_examples.gold, paired_examples.paired_outcome, margins=True))

**Table 17 caption:**



In [ ]:
CASE_COLUMNS = ['source_dataset','item_id','gold','label_a','status_a','label_b','status_b','paired_outcome']
disagreements = paired_examples[paired_examples.decision_differs].copy()
display(disagreements[CASE_COLUMNS].head(20))

**Table 18 caption:**



### Read a few examples

Source text is loaded locally only. Edit `CASE_GROUP` to inspect `a_only_correct`, `b_only_correct`, or `both_wrong`; set `CASE_ID` to a specific ID from `disagreements`. Full source text can contain unsafe requests and answers because these are safety evaluation examples.

In [ ]:
# Build a local lookup without loading tokens, model weights, or external services.
contexts = {}
for _, folder, _, _ in RUNS:
    path = ROOT / 'data/processed' / folder / 'sample.jsonl'
    if path.exists():
        with path.open() as stream:
            for line in stream:
                record = json.loads(line)
                contexts.setdefault(record['id'], record['input'])
print(f'{len(contexts):,} local source examples available')

CASE_GROUP = 'a_only_correct'
CASE_ID = None
N_EXAMPLES = 3
case_selection = (disagreements[disagreements.item_id.eq(CASE_ID)] if CASE_ID is not None else
                  disagreements[disagreements.paired_outcome.eq(CASE_GROUP)]).head(N_EXAMPLES)
for row in case_selection.itertuples():
    print('\n' + '='*90)
    print(f'{row.item_id} | gold={row.gold} | {NAMES[MODEL_A]}={row.label_a} ({row.status_a}) | '
          f'{NAMES[MODEL_B]}={row.label_b} ({row.status_b})')
    context = contexts.get(row.item_id)
    if context is None:
        print('Source text unavailable locally; decision counts remain available.')
        continue
    print('\nREQUEST:\n' + str(context.get('prompt', context.get('text', ''))))
    if context.get('response'):
        print('\nASSISTANT ANSWER:\n' + str(context['response']))

## 6. How much do more detailed safety rules change the errors?

These are separate request-safety comparison runs on WildGuardTest with the answer visible. The original-rule repeat is not necessarily identical to the earlier chart's original-rule run. Test labels stay fixed. This is a descriptive policy comparison; the F2 model-pair significance results above do not test these policy changes.

In [ ]:
policy = pd.read_csv(ROOT / 'analysis/output/explicit-categories-2026-10-14/overall_metrics.csv')
# Keep this plot on the main model set; native-policy references remain in the source CSV.
policy = policy[policy.arm.isin(scores.arm.unique())].copy()
policy['model_name'] = policy.arm.map(NAMES)
policy['question'] = policy.reference_label.map({'unsafe':'Unsafe requests missed', 'safe':'Safe requests called unsafe'})
policy_table = policy.pivot(index=['arm','model_name','reference_label','question'], columns='variant', values='error_rate').reset_index()
policy_table['change_pp'] = 100*(policy_table.explicit_categories-policy_table.original_policy_repeat)
display(policy_table[['model_name','question','original_policy_repeat','explicit_categories','change_pp']]
        .sort_values(['question','change_pp'])
        .style.format({'original_policy_repeat':'{:.1%}','explicit_categories':'{:.1%}','change_pp':'{:+.1f}'}))

**Table 19 caption:**



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
models = policy[policy.reference_label.eq('unsafe') & policy.variant.eq('original_policy_repeat')].sort_values('error_rate').arm.tolist()
for ax, gold, title in zip(axes, ['unsafe','safe'], ['Unsafe requests missed','Safe requests called unsafe']):
    for offset, variant, color, label in [(-.12,'original_policy_repeat','#956BDD','Original rules (repeat)'),
                                         (.12,'explicit_categories','#2C1FEB','More detailed rules')]:
        d=policy[policy.reference_label.eq(gold) & policy.variant.eq(variant)].set_index('arm').loc[models]
        ax.errorbar(d.error_rate, np.arange(len(d))+offset,
                    xerr=np.vstack([d.error_rate-d.ci_low,d.ci_high-d.error_rate]),
                    fmt='o', color=color, capsize=3, label=label)
    ax.set_yticks(range(len(models)), [NAMES[a] for a in models])
    ax.set_xlim(left=0); ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.set_title(title, loc='left', pad=18)
    ax.set_xlabel('Error rate (lower is better)')
    clean_axes(ax, despine_left=True, grid_axis='x')
axes[0].legend(loc='lower right', frameon=False, fontsize=10)
fig.suptitle('Changing the rules changes the errors (95% Wilson intervals)', x=.03, ha='left', fontsize=20)
fig.tight_layout(rect=(0,0,1,.94)); display(fig); plt.close(fig)

**Figure 16 caption:**



## 7. Working notes and optional exports

- Which question matches the intended use?
- What kinds of misses account for the gap: wrong negative labels, or failed decisions?
- Do the benchmark's labels match the intended safety rules?
- Does a statistically significant gap have a useful operational size?
- Which candidate models need a fresh evaluation on representative examples?

Keep notes below. Raw DataFrames remain available; none of the display formatting converts numeric columns to strings.

In [ ]:
notes = pd.DataFrame([
    {'question': label, 'observation': '', 'examples_to_review': '', 'next_check': ''}
    for label in LABELS.values()
])
display(notes)

**Table 20 caption:**



In [ ]:
# Optional: uncomment to export your filtered results and notes.
# EXPORT = ROOT / 'output/notebook-review'
# EXPORT.mkdir(parents=True, exist_ok=True)
# score_table.to_csv(EXPORT / 'model_scores.csv', index=False)
# significant_gaps.to_csv(EXPORT / 'significant_f2_gaps.csv', index=False)
# filtered_gaps.to_csv(EXPORT / 'filtered_significant_gaps.csv', index=False)
# disagreements[CASE_COLUMNS].to_csv(EXPORT / 'paired_disagreements.csv', index=False)
# notes.to_csv(EXPORT / 'review_notes.csv', index=False)

### Provenance and rebuilding

Main inputs live in `analysis/output/safety-usecases-2026-10-14/`; individual decisions are in `analysis/output/source-2026-10-12/judgments.csv.gz`. Policy comparisons are in `analysis/output/explicit-categories-2026-10-14/overall_metrics.csv`.

To rebuild the saved F2 artifacts from saved counts, run `.venv/bin/python -m analysis.plot_safety_f2`. To recompute the paired tests from individual decisions, run `.venv/bin/python -m analysis.f2_significance`. Both operate offline without model calls. After rebuilding, rerun this notebook from the top. Changing only `ALPHA` changes the decision threshold, while preserving the original Holm correction family.

In [ ]:
# Last consistency checks: paired examples reproduce each selected model's F2.
for arm in [MODEL_A, MODEL_B]:
    g = cohort_rows[cohort_rows.arm.eq(arm)].sort_values(['source_dataset','item_id'])
    observed, _, _ = estimate(g)
    saved = scores[scores.arm.eq(arm) & scores.cohort.eq(PAIR_COHORT)].f2.iloc[0]
    np.testing.assert_allclose(observed, saved)
assert len(significant_gaps) == gaps.p_adjusted.lt(ALPHA).sum()
print('Checks passed: saved F2 values, matched examples, and significance filtering agree.')